# Create schema

In [0]:
CATALOG = "dbr_dev_ua5816bd"
SCHEMA = "mialkovska_viktor594"
VOLUME = "raw_files"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATASET_PATH = f"{RAW_PATH}/smart_shipment_route_monitoring"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
import os

DATASET = "tharishreddy22/smart-shipment-logistics-and-route-event-monitoring"

if os.path.exists(DATASET_PATH) and len(os.listdir(DATASET_PATH)) > 0:
    print("Dataset already exists. Download skipped.")
print(os.listdir(DATASET_PATH))

In [0]:
DATA_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_files/"
    "smart_shipment_route_monitoring/shipment"
)

SHIPMENTS_PATH = f"{DATA_PATH}/shipments_master.csv"

print(SHIPMENTS_PATH)

In [0]:
shipments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SHIPMENTS_PATH)
)


display(shipments_df.limit(10))

In [0]:
shipments_df.printSchema()


In [0]:
SHIPMENTS_RAW = f"{CATALOG}.{SCHEMA}.shipments_raw"

(
    shipments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SHIPMENTS_RAW)
)


In [0]:
shipments_df = spark.table(SHIPMENTS_RAW)

origins = shipments_df.select("origin_port")
destinations = shipments_df.select("destination_port")

locations_df = (
    origins
    .withColumnRenamed("origin_port", "location_name")
    .union(
        destinations.withColumnRenamed("destination_port", "location_name")
    )
    .dropna()
    .distinct()
)

print(f"Distinct locations: {locations_df.count()}")

display(locations_df)

In [0]:
import requests

def get_location_info(location_name):

    try:
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location_name.replace("_", " "), "count": 1},
            timeout=20
        )

        data = response.json()

        if "results" not in data:
            return {
                "location_name": location_name,
                "country": None,
                "country_code": None
            }

        result = data["results"][0]

        return {
            "location_name": location_name,
            "country": result.get("country"),
            "country_code": result.get("country_code")
        }

    except Exception as e:
        print("Failed:", location_name, e)

        return {
            "location_name": location_name,
            "country": None,
            "country_code": None
        }

In [0]:
import time

locations = [
    row["location_name"]
    for row in locations_df.collect()
]

location_results = []

for i, location in enumerate(locations, start=1):
    
    result = get_location_info(location)
    location_results.append(result)
    
    print(f"{location} -> {result['country']}")
    
    time.sleep(0.2)

In [0]:
from pyspark.sql.types import StringType, StructField, StructType

location_schema = StructType([
    StructField("location_name", StringType(), False),
    StructField("country", StringType(), True),
    StructField("country_code", StringType(), True)
])

location_df = spark.createDataFrame(
    location_results,
    schema=location_schema
)

display(location_df)


In [0]:
from pyspark.sql import functions as F

origin_df = location_df.select(
    F.col("location_name").alias("origin_port"),
    F.col("country").alias("origin_country")
)

destination_df = location_df.select(
    F.col("location_name").alias("destination_port"),
    F.col("country").alias("destination_country")
)

In [0]:
shipment_analysis_df = (
    shipments_df
    .join(origin_df, on="origin_port", how="left")
    .join(destination_df, on="destination_port", how="left")
)

shipment_analysis_df.printSchema()

In [0]:
shipment_analysis_df = shipment_analysis_df.select(
    "shipment_id",
    "carrier",
    "origin_port",
    "origin_country",
    "destination_port",
    "destination_country",
    "transport_mode",
    "status",
    "distance_km",
    "freight_cost_usd",
    "delay_hours",
    "risk_score"
)

display(shipment_analysis_df.limit(10))

In [0]:

shipment_analysis_df = shipment_analysis_df.withColumn(
    "status",
    F.regexp_replace("status", "_", " ")
)

In [0]:
FINAL_TABLE = f"{CATALOG}.{SCHEMA}.shipment_analysis"

(
    shipment_analysis_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FINAL_TABLE)
)

print(f"Created: {FINAL_TABLE}")

In [0]:
status_analysis_df = (
    shipment_analysis_df
    .groupBy("status")
    .agg(F.count("*").alias("shipments"))
    .orderBy(F.desc("shipments"))
)

display(status_analysis_df)

In [0]:
transport_delay_df = (
    shipment_analysis_df
    .groupBy("transport_mode")
    .agg(
        F.round(F.avg("delay_hours"), 2)
        .alias("avg_delay_hours")
    )
    .orderBy(F.desc("avg_delay_hours"))
)

display(transport_delay_df)

In [0]:
delayed_shipments_df = (
    shipment_analysis_df
    .filter(F.col("delay_hours") > 0)
    .select(
        "shipment_id",
        "carrier",
        "origin_country",
        "destination_country",
        "transport_mode",
        "delay_hours",
        "risk_score"
    )
)

display(delayed_shipments_df.limit(10))